<a href="https://colab.research.google.com/github/Shamsfathalla/FlyRank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shamsfathalla/FlyRank-Starter-Notebooks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building the matrix: I am pulling the daily data for March 2026. Because a missing value in analytics usually means "zero events happened," I fill missing traffic/impression metrics with 0.

For ranking position, a missing value means the page didn't rank, so I fill it with a penalty value of 100. Finally, I engineer a new feature: engagement_rate (engaged sessions / total sessions) to capture quality regardless of volume.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
table_path = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
print("Connection established.")

Connection established.


In [ ]:
# Pull raw data
query = f"""
SELECT
    gsc_impressions, gsc_clicks, gsc_avg_position,
    ga4_sessions, ga4_engaged_sessions, scroll_events
FROM {table_path}
WHERE month = '2026-03'
LIMIT 50000
"""
df = con.sql(query).df()
print(f"Data pulled: {df.shape[0]} rows.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data pulled: 50000 rows.


In [ ]:
# Create the Target (Opportunity = High impressions, exactly 0 clicks)
y = ((df['gsc_impressions'] > 50) & (df['gsc_clicks'] == 0)).astype(int)
print(f"Target created. Found {y.sum()} opportunity pages in this slice.")

Target created. Found 4079 opportunity pages in this slice.


In [ ]:
# Handle Missing Values (Imputation)
df['gsc_impressions'] = df['gsc_impressions'].fillna(0)
df['ga4_sessions'] = df['ga4_sessions'].fillna(0)
df['ga4_engaged_sessions'] = df['ga4_engaged_sessions'].fillna(0)
df['scroll_events'] = df['scroll_events'].fillna(0)
df['gsc_avg_position'] = df['gsc_avg_position'].fillna(100)

# Engineer Features
df['engagement_rate'] = np.where(
    df['ga4_sessions'] > 0,
    df['ga4_engaged_sessions'] / df['ga4_sessions'],
    0
)
print("Missing values handled and engagement_rate engineered.")

Missing values handled and engagement_rate engineered.


In [ ]:
# Final Feature Vector (Drop the leakage column and base components)
X = df[['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'scroll_events', 'engagement_rate']].copy()

print(f"Feature vector built: {X.shape[0]} rows, {X.shape[1]} features.")
display(X.head(3))

Feature vector built: 50000 rows, 5 features.


,gsc_impressions,gsc_avg_position,ga4_sessions,scroll_events,engagement_rate
0,20,3.350,0,0,0.0
1,1,0.000,0,0,0.0
2,125,4.928,0,0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

gsc_impressions: Search visibility volume. Missing filled with 0 (no views). Available day-of.

gsc_avg_position: Search ranking depth. Missing filled with 100 (unranked penalty). Available day-of.

ga4_sessions: Total visits. Missing filled with 0. Available day-of.

scroll_events: Page interaction depth. Missing filled with 0. Available day-of.

engagement_rate: Engineered metric (engaged / total). Calculated securely to avoid division by zero. Available day-of.

No categorical variables required encoding for this specific scoring frame.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature Vector Integrity Check:")
# Verify no nulls remain and show data types
info_df = pd.DataFrame({
    'Data Type': X.dtypes,
    'Missing Values': X.isnull().sum(),
    'Min Value': X.min(),
    'Max Value': X.max()
})
display(info_df)

Feature Vector Integrity Check:


,Data Type,Missing Values,Min Value,Max Value
gsc_impressions,int64,0,0.0,6912.0
gsc_avg_position,float64,0,0.0,142.0
ga4_sessions,Int64,0,0.0,24.0
scroll_events,Int64,0,0.0,9.0
engagement_rate,float64,0,0.0,1.0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

To prove my features are safe, I will run a correlation check between my features and my target label.
If a feature causes a massive, unnatural correlation spike (like > 0.8 or < -0.8), it usually means the feature secretly contains the answer. gsc_impressions will naturally have a slight positive correlation because it is part of the baseline filter (> 50), but the rest should show realistic, modest relationships.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Attack the features: Check Pearson correlation with the target
correlations = X.apply(lambda col: col.corr(y)).sort_values(ascending=False)

print("Leakage Hunt: Correlation with Target (is_opportunity)")
for feat, corr in correlations.items():
    status = "WARNING (Possible Leak)" if abs(corr) > 0.5 else "Safe"
    print(f"{feat:<20}: {corr:>7.4f}  {status}")

Leakage Hunt: Correlation with Target (is_opportunity)
gsc_impressions     :  0.2677  Safe
ga4_sessions        : -0.0004  Safe
engagement_rate     : -0.0051  Safe
scroll_events       : -0.0117  Safe
gsc_avg_position    : -0.3703  Safe


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

gsc_clicks: Excluded because it creates 100% target leakage. The label is strictly defined by clicks being 0.

client_hash_id & content_hash_id: Excluded because they are identifiers. A machine learning model should learn behavior, not memorize specific URLs or clients.

report_date & month: Excluded because time-series dates can cause the model to memorize seasonality or specific days rather than learning the page's actual signal.

ga4_engaged_sessions: Excluded as a raw feature because I converted it into the more robust engagement_rate ratio to prevent multi-collinearity with total sessions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.